# Are diseases in different therapeutic areas more independent than diseases in the same one?

This is the assumption behind gps_TA, and R2-MJ-12 disputes it directly:

> "Diseases linked through a single cluster are likely to be pathologically correlated; I am not
> convinced that spread across therapeutic areas captures real diversity."

The genetic-correlation matrix S answers it without any modelling. Take S's disease traits, map each to
one therapeutic area, and compare the genetic correlation of **within-area** disease pairs against
**between-area** pairs. If spread across areas is meaningless, the two distributions coincide.

## Four statistics, chosen to be simple to report

1. **Mean and median |r<sub>g</sub>|**, within-area versus between-area, with a **permutation P value** —
   the area labels are shuffled across diseases, because disease pairs share diseases and are not
   independent observations, so a t-test would be invalid.
2. **Probability of superiority** — pick one within-area pair and one between-area pair at random; how
   often is the within-area pair the more correlated? This is Mann–Whitney *U* / (n₁n₂), and it needs no
   distributional assumption. 0.5 means no difference.
3. **Tail fractions** — the share of pairs exceeding |r<sub>g</sub>| = 0.2, 0.3, 0.5, in each group.
   Redundancy is driven by the tail, and a fold-difference in the tail is the most quotable form.
4. **Effective-number efficiency** — for each area with k ≥ 2 diseases, the Li & Ji Meff of that area's
   submatrix divided by k, against **size-matched random disease sets drawn across areas**. This is the
   manuscript's claim in its own currency: if an area's k diseases are worth fewer independent traits
   than k diseases picked at random, then counting areas is closer to counting independent signals than
   counting diseases.

A referee's obvious objection — "you have only shown that subtypes of the same disease correlate" — is
pre-empted by repeating everything with **ontology parent/child pairs removed**.

In [1]:
import sys

import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, "../effective-independent-traits")
from eit_lib import meff_li_ji

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)

INTERMEDIATE = "../../../data/intermediate_files/"
RELEASE = "../../../data/25.06/"
SEED = 20260813
N_PERMUTATIONS = 10000
N_MATCHED_DRAWS = 1000

## S's disease traits, and the therapeutic-area map

Same one-area-per-disease rule as `../disease-subsampling/`: the `therapy_area_hierarchy` priority order
from `chapters/01-data-preparation/04_qualifying_dataset_generation.ipynb`, first match wins.

In [2]:
THERAPY_AREA_HIERARCHY = {
    "EFO_0001444": "measurement",
    "MONDO_0045024": "cancer or benign tumor",
    "EFO_0005741": "infectious disease",
    "OTAR_0000009": "injury, poisoning or other complication",
    "OTAR_0000014": "pregnancy or perinatal disease",
    "MONDO_0024458": "disorder of visual system",
    "EFO_0000319": "cardiovascular disease",
    "EFO_0009605": "pancreas disease",
    "EFO_0000540": "immune system disease",
    "EFO_0010282": "gastrointestinal disease",
    "OTAR_0000017": "reproductive system or breast disease",
    "EFO_0010285": "integumentary system disease",
    "EFO_0001379": "endocrine system disease",
    "OTAR_0000010": "respiratory or thoracic disease",
    "EFO_0009690": "urinary system disease",
    "OTAR_0000006": "musculoskeletal or connective tissue disease",
    "MONDO_0021205": "disorder of ear",
    "EFO_0005803": "hematologic disease",
    "EFO_0000618": "nervous system disease",
    "MONDO_0002025": "psychiatric disorder",
    "OTAR_0000020": "nutritional or metabolic disease",
    "OTAR_0000018": "genetic, familial or congenital disease",
    "EFO_0003765": "sign or symptom",
}
PRIORITY = [k for k in THERAPY_AREA_HIERARCHY if k != "EFO_0001444"]

S = pd.read_parquet(INTERMEDIATE + "canonical_pairwise_table/rg_processed.parquet")
canonical = pd.read_parquet(
    INTERMEDIATE + "canonical_pairwise_table/canonical_pairwise_table.parquet",
    columns=[
        "diseaseId_1",
        "therapeutic_area_1",
        "traitFromSource_1",
        "diseaseId_2",
        "therapeutic_area_2",
        "traitFromSource_2",
    ],
)
label = (
    pd.concat(
        [
            canonical[["diseaseId_1", "therapeutic_area_1"]].set_axis(["trait", "ta"], axis=1),
            canonical[["diseaseId_2", "therapeutic_area_2"]].set_axis(["trait", "ta"], axis=1),
        ]
    )
    .drop_duplicates("trait")
    .set_index("trait")["ta"]
)

disease_traits = [t for t in S.index if label.get(t) != "measurement"]
print(
    "S traits:",
    len(S.index),
    "| labelled disease:",
    len(disease_traits),
    "| labelled measurement:",
    len(S.index) - len(disease_traits),
)

ontology = pd.read_parquet(RELEASE + "output/disease/disease.parquet", columns=["id", "descendants", "ancestors"])
roots = ontology[ontology["id"].isin(THERAPY_AREA_HIERARCHY)]
descendants = {r.id: set(list(r.descendants) if r.descendants is not None else []) for r in roots.itertuples()}
assert len(descendants) == len(THERAPY_AREA_HIERARCHY)


def single_area(term):
    areas = {root for root, kids in descendants.items() if term == root or term in kids}
    for root in PRIORITY:
        if root in areas:
            return root
    return "other"


AREA_OF = {t: single_area(t) for t in disease_traits}

S traits: 1114 | labelled disease: 551 | labelled measurement: 563


### Coverage — the limitation that has to be stated first

Most of S's disease traits are FinnGen / PheCode-style MONDO and HP terms that are not descendants of any
of the 22 non-measurement roots, so the hierarchy assigns them `other`. `other` is a residual bucket, not
a therapeutic area, so within-`other` pairs are meaningless and those traits are dropped. The upstream
`therapeutic_area` column in `canonical_pairwise_table.parquet` agrees closely, which shows this is a
property of the data rather than of this mapping.

In [3]:
coverage = pd.DataFrame(
    [
        {
            "source": "this notebook's hierarchy map",
            "other": sum(1 for t in disease_traits if AREA_OF[t] == "other"),
            "total": len(disease_traits),
        },
        {
            "source": "upstream therapeutic_area column",
            "other": int((label.reindex(disease_traits) == "other").sum()),
            "total": len(disease_traits),
        },
    ]
)
coverage["fraction_other"] = (coverage["other"] / coverage["total"]).round(4)
print(coverage.to_string(index=False))

analysis_traits = [t for t in disease_traits if AREA_OF[t] != "other"]
print(f"\ndiseases carrying a real therapeutic area: {len(analysis_traits)} of {len(disease_traits)}")

area_counts = pd.Series({t: THERAPY_AREA_HIERARCHY[AREA_OF[t]] for t in analysis_traits}).value_counts()
distribution = area_counts.rename("n_diseases").rename_axis("therapeutic_area").reset_index()
distribution["within_area_pairs"] = distribution["n_diseases"] * (distribution["n_diseases"] - 1) // 2
distribution.to_csv(INTERMEDIATE + "ta_independence_distribution-r1.csv", index=False)
print()
print(distribution.to_string(index=False))
print(
    f"\nareas represented: {len(distribution)} of 22 non-measurement areas"
    f" | with >= 2 diseases: {int((distribution['n_diseases'] >= 2).sum())}"
)

                          source  other  total  fraction_other
   this notebook's hierarchy map    151    551          0.2740
upstream therapeutic_area column    114    551          0.2069

diseases carrying a real therapeutic area: 400 of 551

                            therapeutic_area  n_diseases  within_area_pairs
                      cancer or benign tumor          59               1711
                      cardiovascular disease          45                990
                    gastrointestinal disease          40                780
musculoskeletal or connective tissue disease          30                435
                      nervous system disease          29                406
                          infectious disease          23                253
                       immune system disease          22                231
                             sign or symptom          22                231
                   disorder of visual system          20               

### Why so many land in `other` — two distinct causes, and one of them is a feature

`other` is not a mapping failure spread evenly over real diseases. Splitting the 342 by *why* they fail
shows the disease side of S is largely a UK Biobank / PheCode PheWAS catalogue rather than a curated
disease list.

In [4]:
in_ontology = set(ontology["id"])


def reason(term):
    if AREA_OF[term] != "other":
        return "mapped to a real area"
    return "absent from the disease ontology" if term not in in_ontology else "in ontology, under no area root"


trait_name = (
    pd.concat(
        [
            canonical[["diseaseId_1", "traitFromSource_1"]].set_axis(["trait", "name"], axis=1),
            canonical[["diseaseId_2", "traitFromSource_2"]].set_axis(["trait", "name"], axis=1),
        ]
    )
    .drop_duplicates("trait")
    .set_index("trait")["name"]
)

audit = pd.DataFrame(
    {
        "trait": disease_traits,
        "prefix": [t.split("_")[0] for t in disease_traits],
        "reason": [reason(t) for t in disease_traits],
        "name": [trait_name.get(t, "") for t in disease_traits],
    }
)
breakdown = pd.crosstab(audit["prefix"], audit["reason"], margins=True)
breakdown.to_csv(INTERMEDIATE + "ta_independence_other_breakdown-r1.csv")
print(breakdown.to_string())

# Every MONDO term that IS in the ontology maps; the failures are purely index membership.
mondo = audit[audit["prefix"] == "MONDO"]
print(
    f"\nMONDO: {len(mondo)} terms, {int((mondo['reason'] == 'mapped to a real area').sum())} mapped,"
    f" {int((mondo['reason'] == 'absent from the disease ontology').sum())} absent from the ontology,"
    f" {int((mondo['reason'] == 'in ontology, under no area root').sum())} present but unmapped"
)

phecode = audit["name"].str.contains("PheCode", case=False, na=False)
questionnaire = audit["name"].str.contains(
    "UKB data field|Illnesses of father|Illnesses of mother|Illnesses of siblings|acceptance"
    "|Have you ever|usually taken|Why stopped|side of head",
    case=False,
    na=False,
)
print(
    f"\ntraits whose source name contains 'PheCode': {int(phecode.sum())} of {len(audit)}"
    f" ({int((phecode & (audit['reason'] == 'mapped to a real area')).sum())} of them still map)"
)
print(
    f"UKB questionnaire / family-history items: {int(questionnaire.sum())} of {len(audit)}"
    " -- these are not diseases at all"
)
print("\nexamples of the questionnaire items:")
for name in audit.loc[questionnaire, "name"].head(6):
    print("   ", name[:78])

reason  in ontology, under no area root  mapped to a real area  All
prefix                                                             
EFO                                  68                    291  359
GO                                    6                      0    6
HP                                   74                     14   88
MONDO                                 0                     95   95
MP                                    1                      0    1
OBA                                   2                      0    2
All                                 151                    400  551

MONDO: 95 terms, 95 mapped, 0 absent from the ontology, 0 present but unmapped

traits whose source name contains 'PheCode': 155 of 551 (91 of them still map)
UKB questionnaire / family-history items: 35 of 551 -- these are not diseases at all

examples of the questionnaire items:
    Have you ever been pregnant?
    Ever had hysterectomy - womb removed (UKB data field 3591)
    Usual

So the 342 exclusions break down as **222 terms with no row in the disease ontology at all** (210 of them
MONDO — and note that *every* MONDO term that is in the ontology does map, so this is purely index
membership, not a hierarchy problem) plus **120 terms that are in the ontology but under no therapeutic-area
root** (76 HP phenotype codes, 37 EFO, 6 GO, 1 MP).

Underneath that: **266 of the 498 (53%) are PheCode-derived** and only 92 of those map, while **52 are UK
Biobank questionnaire or family-history fields** — "Illnesses of father: Heart disease", "Coffee consumed",
"Usual side of head for mobile phone use" — which are not diseases in any sense. Dropping them is
therefore **not only a coverage cost but a purification**: the 156 retained traits are the curated disease
terms, which is the population the therapeutic-area hierarchy was built for.

## Build the pair list

In [5]:
index_of = {t: i for i, t in enumerate(S.index)}
rows = np.array([index_of[t] for t in analysis_traits])
sub = S.values[np.ix_(rows, rows)]
areas = np.array([AREA_OF[t] for t in analysis_traits])
n = len(analysis_traits)

iu, ju = np.triu_indices(n, 1)
rg = sub[iu, ju]
same_area = areas[iu] == areas[ju]

# ontology parent/child flag, for the "you only showed subtypes correlate" objection
ancestors = {
    r.id: set(list(r.ancestors) if r.ancestors is not None else [])
    for r in ontology[ontology["id"].isin(analysis_traits)].itertuples()
}
have_ancestors = sum(1 for t in analysis_traits if t in ancestors)
term_at = np.array(analysis_traits)


def related(i, j):
    a, b = term_at[i], term_at[j]
    return b in ancestors.get(a, set()) or a in ancestors.get(b, set())


nested = np.array([related(i, j) for i, j in zip(iu, ju)])

print(f"disease pairs: {len(rg)} | within-area {int(same_area.sum())} | between-area {int((~same_area).sum())}")
print(f"ancestor lists available for {have_ancestors} of {n} traits")
print(
    f"ontology parent/child pairs: {int(nested.sum())}"
    f" (within-area {int((nested & same_area).sum())}, between-area {int((nested & ~same_area).sum())})"
)
print(
    f"pairs with no measured rg (zero-filled): within {int(((rg == 0) & same_area).sum())},"
    f" between {int(((rg == 0) & ~same_area).sum())}"
)

disease pairs: 79800 | within-area 5844 | between-area 73956
ancestor lists available for 400 of 400 traits
ontology parent/child pairs: 1007 (within-area 741, between-area 266)
pairs with no measured rg (zero-filled): within 15, between 26


## Statistics 1–3, with a permutation null

In [6]:
def compare(keep, label_text, draws=N_PERMUTATIONS, rng_seed=SEED):
    """Within versus between comparison on the pair subset `keep`, with a label-permutation P value.

    The permutation shuffles the therapeutic area over *diseases*, not over pairs, so the dependence
    between pairs that share a disease is preserved under the null. All subsetting happens here, against
    the full pair list, so the observed and permuted statistics always use identical rows.
    """
    values = np.abs(rg[keep])
    group = same_area[keep]
    a, b = values[group], values[~group]
    u = stats.mannwhitneyu(a, b, alternative="greater")
    observed = float(a.mean() - b.mean())

    rng = np.random.default_rng(rng_seed)
    extreme = valid = 0
    for _ in range(draws):
        shuffled = rng.permutation(areas)
        g = (shuffled[iu] == shuffled[ju])[keep]
        if g.sum() == 0 or (~g).sum() == 0:
            continue
        valid += 1
        if abs(values[g].mean() - values[~g].mean()) >= abs(observed):
            extreme += 1

    row = {
        "comparison": label_text,
        "n_pairs": int(keep.sum()),
        "n_within": len(a),
        "n_between": len(b),
        "mean_abs_rg_within": float(a.mean()),
        "mean_abs_rg_between": float(b.mean()),
        "mean_difference": observed,
        "median_abs_rg_within": float(np.median(a)),
        "median_abs_rg_between": float(np.median(b)),
        "probability_of_superiority": float(u.statistic / (len(a) * len(b))),
        "mannwhitney_p": float(u.pvalue),
        "permutation_p": (extreme + 1) / (valid + 1),
        "n_valid_permutations": valid,
    }
    for threshold in (0.2, 0.3, 0.5):
        fa, fb = float((a >= threshold).mean()), float((b >= threshold).mean())
        row[f"frac_ge_{threshold}_within"] = fa
        row[f"frac_ge_{threshold}_between"] = fb
        row[f"fold_ge_{threshold}"] = fa / fb if fb > 0 else np.nan
    return row


all_pairs = np.ones(len(rg), dtype=bool)
comparisons = pd.DataFrame(
    [
        compare(all_pairs, "all disease pairs"),
        compare(~nested, "excluding ontology parent/child pairs"),
        compare(rg != 0, "measured pairs only (drop zero-filled)"),
    ]
)
comparisons.to_csv(INTERMEDIATE + "ta_independence_comparisons-r1.csv", index=False)
print(
    comparisons[
        [
            "comparison",
            "n_within",
            "n_between",
            "mean_abs_rg_within",
            "mean_abs_rg_between",
            "mean_difference",
            "probability_of_superiority",
            "permutation_p",
        ]
    ]
    .round(4)
    .to_string(index=False)
)
print()
print(
    comparisons[
        [
            "comparison",
            "median_abs_rg_within",
            "median_abs_rg_between",
            "frac_ge_0.2_within",
            "frac_ge_0.2_between",
            "fold_ge_0.2",
            "frac_ge_0.5_within",
            "frac_ge_0.5_between",
            "fold_ge_0.5",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

                            comparison  n_within  n_between  mean_abs_rg_within  mean_abs_rg_between  mean_difference  probability_of_superiority  permutation_p
                     all disease pairs      5844      73956              0.4009               0.3166           0.0843                      0.5737         0.0001
 excluding ontology parent/child pairs      5103      73690              0.3898               0.3164           0.0735                      0.5634         0.0001
measured pairs only (drop zero-filled)      5829      73930              0.4019               0.3167           0.0852                      0.5750         0.0001

                            comparison  median_abs_rg_within  median_abs_rg_between  frac_ge_0.2_within  frac_ge_0.2_between  fold_ge_0.2  frac_ge_0.5_within  frac_ge_0.5_between  fold_ge_0.5
                     all disease pairs                0.2962                 0.2045              0.6104               0.5070       1.2039              0.3309      

## Statistic 4 — effective-number efficiency

For each area with k ≥ 2 diseases, Meff / k on that area's submatrix, against 1,000 random disease sets of
the same size k drawn from the whole pool (which will mostly span areas). A ratio below the matched null
means the area's diseases are more redundant than an arbitrary set of the same size.

In [7]:
rng = np.random.default_rng(SEED)
rows_eff, matched_cache = [], {}
for area, group in pd.Series(areas, index=range(n)).groupby(pd.Series(areas, index=range(n))):
    idx = group.index.to_numpy()
    k = len(idx)
    if k < 2:
        continue
    observed = meff_li_ji(sub[np.ix_(idx, idx)])
    if k not in matched_cache:
        draws = []
        for _ in range(N_MATCHED_DRAWS):
            pick = rng.choice(n, size=k, replace=False)
            draws.append(meff_li_ji(sub[np.ix_(pick, pick)]))
        matched_cache[k] = np.array(draws)
    null = matched_cache[k]
    rows_eff.append(
        {
            "therapeutic_area": THERAPY_AREA_HIERARCHY[area],
            "k_diseases": k,
            "meff_within_area": observed,
            "efficiency_within_area": observed / k,
            "meff_matched_null_mean": float(null.mean()),
            "efficiency_matched_null": float(null.mean() / k),
            "null_pct2.5": float(np.quantile(null, 0.025) / k),
            "null_pct97.5": float(np.quantile(null, 0.975) / k),
            "p_one_sided": float(((null <= observed).sum() + 1) / (len(null) + 1)),
        }
    )
efficiency = pd.DataFrame(rows_eff).sort_values("k_diseases", ascending=False)
efficiency.to_csv(INTERMEDIATE + "ta_independence_efficiency-r1.csv", index=False)
print(efficiency.round(4).to_string(index=False))

                            therapeutic_area  k_diseases  meff_within_area  efficiency_within_area  meff_matched_null_mean  efficiency_matched_null  null_pct2.5  null_pct97.5  p_one_sided
                      cancer or benign tumor          59           63.8164                  1.0816                 61.8558                   1.0484       0.9620        1.1366       0.7682
                      cardiovascular disease          45           39.4655                  0.8770                 46.3511                   1.0300       0.9349        1.1256       0.0020
                    gastrointestinal disease          40           36.1440                  0.9036                 41.0018                   1.0250       0.9243        1.1239       0.0140
musculoskeletal or connective tissue disease          30           29.3540                  0.9785                 30.1460                   1.0049       0.8953        1.1208       0.3327
                      nervous system disease          29    

In [8]:
weights = efficiency["k_diseases"]
summary = pd.DataFrame(
    [
        {
            "n_areas_tested": len(efficiency),
            "areas_less_independent_than_matched_null": int(
                (efficiency["efficiency_within_area"] < efficiency["efficiency_matched_null"]).sum()
            ),
            "areas_p_lt_05": int((efficiency["p_one_sided"] < 0.05).sum()),
            "mean_efficiency_within_area": float(efficiency["efficiency_within_area"].mean()),
            "mean_efficiency_matched_null": float(efficiency["efficiency_matched_null"].mean()),
            "weighted_efficiency_within_area": float(np.average(efficiency["efficiency_within_area"], weights=weights)),
            "weighted_efficiency_matched_null": float(
                np.average(efficiency["efficiency_matched_null"], weights=weights)
            ),
            "wilcoxon_p_paired": float(
                stats.wilcoxon(efficiency["efficiency_within_area"], efficiency["efficiency_matched_null"]).pvalue
            ),
        }
    ]
)
summary.to_csv(INTERMEDIATE + "ta_independence_efficiency_summary-r1.csv", index=False)
print(summary.T.to_string())

                                                  0
n_areas_tested                            21.000000
areas_less_independent_than_matched_null  14.000000
areas_p_lt_05                              6.000000
mean_efficiency_within_area                0.899213
mean_efficiency_matched_null               0.970897
weighted_efficiency_within_area            0.922425
weighted_efficiency_matched_null           0.998222
wilcoxon_p_paired                          0.011347


## Exports

| File | Contents |
| ---- | -------- |
| `ta_independence_distribution-r1.csv` | number of S diseases per therapeutic area, and within-area pair counts |
| `ta_independence_other_breakdown-r1.csv` | why each of the 498 disease traits does or does not get an area, by ontology prefix |
| `ta_independence_comparisons-r1.csv` | within versus between |r<sub>g</sub>|, superiority, tail fractions, permutation P — three pair sets |
| `ta_independence_efficiency-r1.csv` | Meff / k per area against a size-matched cross-area null |
| `ta_independence_efficiency_summary-r1.csv` | how many areas fall below the null, and the weighted averages |

Interpretation is in the README.